# Module 2: Single Agent

Build one **Strategy Advisor** agent with three business intelligence tools, run it on focused questions, then hand it the full decision brief and watch where a single agent hits its ceiling.

**Use case:** Throughout this workshop you work with MarketNest, a fictional e-commerce company (2M active users) deciding whether to launch a Premium Subscription Tier. Leadership has three options:

- **Option A:** Exclusive Premium, invite-only for the top 10% of spenders, $19.99/mo
- **Option B:** Gradual Rollout, a 5% A/B pilot with a kill-switch, $14.99/mo
- **Option C:** Full Launch, open to all users, $12.99/mo + 30-day free trial

The same decision brief (`samples/shared/decision_brief.py`) is used in Modules 2, 3 and 4, so you can compare how each design handles it.

**Prerequisites:** Python 3.10+, AWS credentials with Amazon Bedrock access. Module 1 is an optional primer on the agent loop.

---

## Step-1: A first Strands agent

Before the Strategy Advisor, here is the smallest possible Strands agent: a system prompt, one tool, and one call. The pattern — `Agent(system_prompt=..., tools=[...])`, then `agent("...")` — is exactly what the rest of this module builds on.

First install the requirements:


In [ ]:
%pip install -r requirements.txt

Then define a trivial `@tool` and hand it to an `Agent`. The model reads the docstring, decides to call the tool, and the tool's return string flows back into the agent loop:


In [ ]:
from strands import Agent, tool


@tool
def bat_signal() -> str:
    """Light the Bat-Signal over Gotham to call for Batman."""
    print("🦇")
    return "The Bat-Signal shines over Gotham. Help is on the way."


agent = Agent(system_prompt="You are Batman.", tools=[bat_signal])
agent("Use your bat-signal.")


---

## Step-2: Load the tools and the decision brief

`decision_brief_tools.py` (in this folder) holds three mock `@tool` functions over hardcoded MarketNest data, so every run sees the same numbers. Modules 3 and 4 import the same file.

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────
# No model= is passed, so every Agent uses the Strands default on Amazon Bedrock:
# Claude Sonnet 4.6 through the global inference profile (global.anthropic.claude-sonnet-4-6).
# To use another model enabled in your account, create one and pass model=model to each Agent:
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")  # faster
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")  # Amazon Nova: bare ID (the workshop allows no Amazon inference profiles)

import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), "..", "shared"))

from strands import Agent
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data
from decision_brief import DECISION_BRIEF

TOOLS = [get_company_data, get_market_benchmarks, get_competitor_data]
for t in TOOLS:
    print(f"{t.tool_name}: {t.tool_spec['description'].splitlines()[0]}")

---

## Step-3: Define the System Prompt

One agent plays every role: it researches with the tools, weighs the options, and writes the briefing. The prompt sets its job; the tool docstrings tell it when to call each tool.

In [ ]:
SYSTEM_PROMPT = '''You are a senior Corporate Strategy Advisor for MarketNest.
Use your tools to look up company data, market benchmarks, and competitor information.
Never guess a number a tool can give you. If a metric is not available, write "Unknown".

For a quick question, answer in a few data-backed sentences.

For a full decision brief, produce an Executive Strategy Briefing with these sections, in order:
1) Recommendation (1-2 lines)
2) Options considered (A/B/C)
3) Tradeoffs: "We gain X; we risk Y."
4) Risks + mitigations
5) Assumptions + unknowns
6) Escalate? (Yes/No + reason; Yes if constraints are unmet or approvals are missing)
7) Next steps (owner + timeline)'''

---

## Step-4: Create the Agent and ask a focused question

In [ ]:
agent = Agent(system_prompt=SYSTEM_PROMPT, tools=TOOLS)

result = agent(
    "What is MarketNest's current CLV and churn rate? "
    "How does that compare to the e-commerce industry benchmark?"
)

---

## Step-5: Inspect the Agent Loop

As in Module 1, every step is stored in `agent.messages`:

```
User message → LLM reasons → selects a tool → tool executes → result appended to context → LLM reasons again → ... repeat until done
```

In [ ]:
import json

print(f"Total messages: {len(agent.messages)}")
print("=" * 65)

for i, msg in enumerate(agent.messages):
    for block in msg["content"]:
        if msg["role"] == "user" and "text" in block:
            print(f"[{i}] USER:         {block['text'][:100]}")
        elif "toolResult" in block:
            for item in block["toolResult"]["content"]:
                if "text" in item:
                    print(f"[{i}] TOOL RESULT:  {item['text'][:100]}")
        elif "toolUse" in block:
            tu = block["toolUse"]
            print(f"[{i}] TOOL CALL:    {tu['name']}({json.dumps(tu['input'])})")
        elif "text" in block:
            print(f"[{i}] ASSISTANT:    {block['text'][:100]}")

In [ ]:
summary = result.metrics.get_summary()
usage = summary["accumulated_usage"]
print("Loop metrics:")
print(f"  Cycles (LLM calls):  {summary['total_cycles']}")
print(f"  Input tokens:        {usage['inputTokens']}")
print(f"  Output tokens:       {usage['outputTokens']}")
for name, data in summary["tool_usage"].items():
    stats = data["execution_stats"]
    print(f"  {name}: calls={stats['call_count']} | avg_time={stats['average_time']:.3f}s")

---

## Step-6: Follow-up Questions

The agent keeps its conversation history, so follow-ups build on what it already looked up.

In [ ]:
agent("What did ShopMart do when they launched their premium tier? How did it perform?");

In [ ]:
agent("And PrimeStore? Which approach, ShopMart's or PrimeStore's, had the better ROI?");

---

## Step-7: The Ceiling: One Agent, One Giant Context

So far the agent handled focused questions well. Now give a fresh agent the **full decision brief**: three options, context, financial targets, a deadline, and a request for a complete briefing.

Watch what happens: one agent researches, analyzes all three options, and writes the recommendation in a single context window.

In [ ]:
print(DECISION_BRIEF)

ceiling_agent = Agent(system_prompt=SYSTEM_PROMPT, tools=TOOLS)
result_ceiling = ceiling_agent(DECISION_BRIEF)

In [ ]:
summary = result_ceiling.metrics.get_summary()
usage = summary["accumulated_usage"]
print("Single agent on the full brief:")
print(f"  Cycles (LLM calls):  {summary['total_cycles']}")
print(f"  Input tokens:        {usage['inputTokens']}")
print(f"  Output tokens:       {usage['outputTokens']}")
print(f"  Messages in context: {len(ceiling_agent.messages)}")
print(f"  Tools called:        {', '.join(summary['tool_usage']) or 'none'}")
print()
print("One agent played Researcher + Option Analyst x3 + Writer.")
print("Every tool result and every option's reasoning share one context window,")
print("and every model call re-reads all of it. That is the ceiling.")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `Agent(tools=[...], system_prompt=...)` | One agent with business intelligence tools |
| Tool docstrings | The model picks which tool to call from the docstring |
| `agent.messages` | Every user turn, LLM response, tool call and tool result |
| `result.metrics.get_summary()` | Cycles, tokens and per-tool stats |
| **The ceiling** | One agent juggling researcher, analyst and writer roles puts everything in one context |

---

## What's Next

In **Module 3: Sequential Chain**, you split this work across three specialist agents, Researcher → Analyst → Synthesizer, each with its own focused context. Graph edges fix the order in which they run.

---

## Want a real multi-turn conversation?

Each notebook cell is a single turn. To chat back and forth with the same agent, run the companion script in a terminal:

```bash
cd samples/02-single-agent
pip install -r requirements.txt
python chat.py
```

Type or paste your message; `quit` or Ctrl+C exits.